파일을 sc.textFile()로 읽어서 다음을 계산하세요.
• 첫 번째 정수 쌍의 첫 번째 원소
• 모든 정수 쌍의 첫 번째 원소 중 최댓값
• 모든 정수 쌍의 첫 번째 원소 중 최솟값
• 첫 번째 원소가 1인 정수 쌍의 개수
• 첫 번째 원소가 2인 정수 쌍의 개수
• 이 다섯 개의 숫자를 모두 더한 뒤 출력하세요.

In [ ]:
import os
import sys
import urllib.request
from pyspark.sql import SparkSession

# 1. [Q1과 동일] Windows VS Code 파이썬 워커 통신 오류 방지 환경변수 및 로컬호스트 설정
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("Q2_IntPair") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()
sc = spark.sparkContext

# 2. 파일 다운로드 및 RDD 로드
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt"
urllib.request.urlretrieve(url, "int-pair-input.txt")

lines = sc.textFile("int-pair-input.txt")

# 3.Parsing: 빈 줄을 제거하고 괄호 '('와 ')' 및 콤마 ','를 모두 제거한 후 정수 리스트로 변환
pairs_rdd = lines.filter(lambda line: line.strip() != "") \
                 .map(lambda line: [int(v) for v in line.replace('(', '').replace(')', '').replace(',', ' ').split()])

# 첫 번째 원소(x)만 모은 RDD 생성
first_elems_rdd = pairs_rdd.map(lambda pair: pair[0])

# 4. 5가지 값 계산 (Spark Action 연산들)
# (1) 첫 번째 정수 쌍의 첫 번째 원소
v1 = pairs_rdd.first()[0]

# (2) 모든 정수 쌍의 첫 번째 원소 중 최댓값
v2 = first_elems_rdd.reduce(lambda a, b: max(a, b))

# (3) 모든 정수 쌍의 첫 번째 원소 중 최솟값
v3 = first_elems_rdd.reduce(lambda a, b: min(a, b))

# (4) 첫 번째 원소가 1인 정수 쌍의 개수
v4 = first_elems_rdd.filter(lambda x: x == 1).count()

# (5) 첫 번째 원소가 2인 정수 쌍의 개수
v5 = first_elems_rdd.filter(lambda x: x == 2).count()

# 5. 다섯 개 숫자 합산
final_answer = v1 + v2 + v3 + v4 + v5

print(f"1. 첫 번째 정수 쌍의 첫 원소: {v1}")
print(f"2. 첫 번째 원소 최댓값: {v2}")
print(f"3. 첫 번째 원소 최솟값: {v3}")
print(f"4. 첫 번째 원소가 1인 개수: {v4}")
print(f"5. 첫 번째 원소가 2인 개수: {v5}")
print(f"\n최종 합계: {final_answer}")

1. 첫 번째 정수 쌍의 첫 원소: 2
2. 첫 번째 원소 최댓값: 32
3. 첫 번째 원소 최솟값: 0
4. 첫 번째 원소가 1인 개수: 31
5. 첫 번째 원소가 2인 개수: 28

최종 합계: 93
